# Lab 12-1 - Logistic Regression

This lab covers the `sklearn` implementation of logistic regression, which is straightforward and largely uses the same syntax as other algorithms we've studied.

In [ ]:
import pandas as pd
import numpy as np

Examples throughout the lab will use the Iowa City home sales data that we've seen in several of our previous labs. Logistic regression needs a *binary* outcome, so we'll create one: whether a home sold for **more than the median sale price** of the training homes (`1` = yes, `0` = no). We have

- n = 621 houses
- p = 11 features used to predict

We leave out `assessed` as a predictor since a home's assessed value is nearly the same thing as its price.

In [ ]:
# Read data
ic = pd.read_csv(filepath_or_buffer = "../data/IowaCityHomeSales.csv")

# Split training and testing sets
from sklearn.model_selection import train_test_split
train, test = train_test_split(ic, test_size = 0.2, random_state = 7)

# Create the binary target: 1 if the home sold for more than the (training) median sale price
median_price = train['sale.amount'].median()
train_y = (train['sale.amount'] > median_price).astype(int)

# Keep only the numeric predictors (though we could use one-hot encoding if we wanted to)
train_X = train.select_dtypes(include = "number").drop(labels = ['sale.amount', 'assessed'], axis = 1)

# Print n and p
print(train_X.shape)

# Print the proportion of homes in each class
print(train_y.value_counts(normalize = True))

## A) Logistic Regression in `sklearn`

Logistic regression is performed using the `LogisticRegression()` function in `sklearn`. Like `LinearRegression()`, it is compatible with pipelines and all the pre-processing tools we've been using:

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Simple pipeline using logistic regression
# (C = np.inf turns off shrinkage of the weights; see the note below)
pipe = Pipeline(steps = [
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(C = np.inf))
])

Just as in linear regression, we can look at each predictor's weight, and the intercept (bias) is stored separately from the slopes:

In [ ]:
# Fit the pipeline to the training data
log_model = pipe.fit(X = train_X, y = train_y)

# Print the estimated intercept (bias)
print(log_model.named_steps['model'].intercept_)

print("\n")

# Print the model's 11 slope weights, labeled by predictor
# (.coef_ is a table with only one row for a binary outcome, so [0] picks out that row)
pd.Series(data = log_model.named_steps['model'].coef_[0], index = train_X.columns)

Note:
- By default `LogisticRegression()` shrinks the estimated weights toward 0. Setting `C = np.inf` turns this shrinkage off, so we get the ordinary logistic regression weights. We'll explain what shrinkage is (and what `C` controls) in a future lecture on regularization.
- The weights are on the *log-odds* scale: the model says $\log\big(\tfrac{\pi}{1-\pi}\big) = w_0 + w_1X_1 + \ldots + w_pX_p$, where $\pi$ is the probability that the outcome is 1. Exponentiating a weight gives its *multiplicative* effect on the **odds** $\tfrac{\pi}{1-\pi}$. For example, $e^{0.44} = 1.55$ means a 1-unit increase in that predictor goes with a 55% increase in the odds.
- Since we used standard scaling, a 1-unit increase means a one-standard-deviation increase in the predictor.

Like linear regression, logistic regression has no tuning parameters, but we can still use a cross-validated grid search by exploring models with and without an intercept (bias). Here the scoring metric is accuracy:

In [ ]:
# Simple grid search
from sklearn.model_selection import GridSearchCV
params = {'model__fit_intercept': [True, False]}

grid_result = GridSearchCV(
    estimator = pipe, 
    param_grid = params, 
    cv = 5, 
    scoring = 'accuracy'
    ).fit(X = train_X, y = train_y)

print(grid_result.best_params_)
print(grid_result.best_score_)

In [ ]:
# Recall this is how we extract the best cross-validated model from above for future use:
grid_result.best_estimator_

## B) Question 1

Suppose we'd like to predict whether a home sold for more than the median sale price using only its size.

- **Part a)**: Create two objects, `train_X_size` and `train_y_size` where the latter records whether each home sold for more than the median sale price (1 = yes) and the former is a matrix containing the predictors: `area.living`, `bedrooms`, and `area.lot`.
- **Part b)**: Create a pipeline `pipe_q1` that includes the following steps:
    - `'scaler'`: standard scaling pre-processing
    - `'transformer'`: Yeo-Johnson normalizing transformation pre-processing on the model's predictors
    - `'model'`: logistic regression model (with no shrinkage, as above)
- **Part c)**: Use a cross-validated grid search with accuracy as the scoring metric to evaluate whether the use of a normalizing transformation on these predictors improves performance relative to an approach that doesn't involve any normalization.
    - Call the space of parameters `GridSearchCV()` considers `params_q1`
    - Save the results of `GridSearchCV()` in `grid_q1_result`
    - Print the best parameter as well as the associated best score
- **Part d)**: Print the estimated weights of the best model in Part c), then exponentiate them. Interpret the exponentiated weight of `area.living` in one sentence. *Hint*: remember the model's weights are on the log-odds scale, and think about what a 1-unit change in a transformed predictor means.

In [ ]:
# Code for Q1.a)



In [ ]:
# Code for Q1.b)



In [ ]:
# Code for Q1.c)



In [ ]:
# Starter Code for Q1.d) assuming you've followed the naming conventions above
weights_q1 = grid_q1_result.best_estimator_.named_steps['model'].coef_[0]

print(dict(zip(train_X_size.columns, weights_q1)))
print(dict(zip(train_X_size.columns, np.exp(weights_q1))))

**Solution for Q1.d)**

